# L07 · Policy Gradient와 REINFORCE

## Goal

- log-derivative trick을 코드로 잇는다
- reward-to-go를 계산한다
- baseline의 역할을 설명한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L07:toy:42").hexdigest()
print(f"lesson=L07 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L07 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:94c087d9fa42e86dbbcd15d9ad9a476a7cc879c6a3e4e6b1ee39caf7c356b141 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: 확률·미분 → **Policy Gradient·REINFORCE** → Actor-Critic·PPO

$$\nabla_\theta J(\theta)=\mathbb{E}\left[\nabla_\theta\log\pi_\theta(a_t\mid s_t)(G_t-b(s_t))\right]$$

log-derivative trick은 sample한 action의 log-prob에 return을 곱해 expectation의 gradient를 추정합니다. reward-to-go는 과거 action에 미래 reward를 배분합니다. action과 무관한 baseline은 기대 gradient를 바꾸지 않으면서 분산을 줄입니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** positive advantage가 곱해진 chosen log-prob의 loss gradient 부호는 무엇인가요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>음수입니다. gradient descent가 log-prob를 키우려면 loss 미분은 음수여야 합니다.</details>

In [2]:
from rl_study.algorithms.policy_gradient import reinforce_loss
from rl_study.algorithms.tabular import monte_carlo_returns
chosen_log_probs = torch.tensor([-0.7, -0.5, -0.2], requires_grad=True)
rewards = torch.tensor([0.0, 0.0, 1.0])
reward_to_go = monte_carlo_returns(rewards, gamma=0.9)
pg_loss = reinforce_loss(chosen_log_probs, reward_to_go, baseline=0.2)
pg_loss.backward()
print({"returns": reward_to_go.tolist(),
       "loss": round(float(pg_loss.detach()), 4),
       "logprob_gradient": chosen_log_probs.grad.tolist()})

{'returns': [0.809999942779541, 0.8999999761581421, 1.0], 'loss': 0.3123, 'logprob_gradient': [-0.20333331823349, -0.23333333432674408, -0.2666666805744171]}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** return과 baseline은 policy parameter 관점에서 detach해야 credit 값이 actor를 통해 다시 만들어지지 않습니다. actor-critic은 learned baseline으로 이를 일반화합니다.

**흔한 함정:** loss 부호를 직관으로만 외우면 maximize/minimize 변환에서 자주 뒤집힙니다. positive advantage의 log-prob gradient가 음수인지 직접 assert합니다. 회귀 test: `test_reinforce_sign`, `test_actor_advantage_detached`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert chosen_log_probs.grad[-1].item() < 0.0
assert reward_to_go[-1].item() == 1.0
print("checks=passed")

checks=passed


**회상 문제:** action에 의존하는 baseline을 빼면 왜 policy-gradient 추정이 편향될 수 있나요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** 세 reward-to-go가 모두 양수였고 chosen log-prob gradient도 모두 음수였습니다. 더 큰 advantage일수록 절댓값이 컸습니다.
- 실제 확인: `test_reinforce_sign`, `test_actor_advantage_detached`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L08에서 baseline을 critic으로 학습하고 GAE로 bias-variance를 조절한 뒤 PPO로 update 폭을 제한합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

[상세 구현 문서](../../docs/algorithms/classic.md) · [강좌 지도](../../docs/course-map.md)

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`